# Movie Expert Agent

OpenAI Agents SDK + Clean Architecture로 구현한 영화 전문가 에이전트.

**API**: `https://nomad-movies.nomadcoders.workers.dev`

| 도구 | 엔드포인트 | 설명 |
|------|-----------|------|
| `get_popular_movies()` | `/movies` | 인기 영화 목록 |
| `get_movie_details(id)` | `/movies/:id` | 영화 상세 정보 |
| `get_movie_credits(id)` | `/movies/:id/credits` | 출연진 및 제작진 |

## 1. Domain Layer — 엔티티 & 포트

In [7]:
from pydantic import BaseModel
from abc import ABC, abstractmethod


# --- Entities ---

class Movie(BaseModel):
    id: int
    title: str
    overview: str | None = None
    vote_average: float | None = None
    release_date: str | None = None
    poster_path: str | None = None


class CastMember(BaseModel):
    name: str
    character: str | None = None
    known_for_department: str | None = None


class CrewMember(BaseModel):
    name: str
    job: str | None = None
    department: str | None = None


class MovieCredits(BaseModel):
    movie_id: int
    cast: list[CastMember]
    crew: list[CrewMember]


# --- Port (Interface) ---

class IMovieRepository(ABC):
    @abstractmethod
    async def get_popular_movies(self) -> list[Movie]: ...

    @abstractmethod
    async def get_movie_details(self, movie_id: int) -> Movie: ...

    @abstractmethod
    async def get_movie_credits(self, movie_id: int) -> MovieCredits: ...

## 2. Application Layer — Use Cases

In [8]:
class GetPopularMoviesUseCase:
    def __init__(self, repo: IMovieRepository):
        self._repo = repo

    async def execute(self) -> list[Movie]:
        return await self._repo.get_popular_movies()


class GetMovieDetailsUseCase:
    def __init__(self, repo: IMovieRepository):
        self._repo = repo

    async def execute(self, movie_id: int) -> Movie:
        return await self._repo.get_movie_details(movie_id)


class GetMovieCreditsUseCase:
    def __init__(self, repo: IMovieRepository):
        self._repo = repo

    async def execute(self, movie_id: int) -> MovieCredits:
        return await self._repo.get_movie_credits(movie_id)

## 3. Infrastructure Layer — API Repository

In [ ]:
import httpx

API_BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"


class MovieAPIRepository(IMovieRepository):
    def __init__(self):
        self._base_url = API_BASE_URL

    async def get_popular_movies(self) -> list[Movie]:
        async with httpx.AsyncClient() as client:
            resp = await client.get(f"{self._base_url}/movies")
            resp.raise_for_status()
            return [Movie(**m) for m in resp.json()]

    async def get_movie_details(self, movie_id: int) -> Movie:
        async with httpx.AsyncClient() as client:
            resp = await client.get(f"{self._base_url}/movies/{movie_id}")
            resp.raise_for_status()
            return Movie(**resp.json())

    async def get_movie_credits(self, movie_id: int) -> MovieCredits:
        async with httpx.AsyncClient() as client:
            resp = await client.get(f"{self._base_url}/movies/{movie_id}/credits")
            resp.raise_for_status()
            data = resp.json()
            # API returns a flat list of cast members (no crew)
            items = data if isinstance(data, list) else data.get("cast", [])
            cast = [CastMember(**c) for c in items[:10]]
            return MovieCredits(movie_id=movie_id, cast=cast, crew=[])

## 4. Adapters Layer — Agent & Tools

In [10]:
from dotenv import load_dotenv
import os
load_dotenv()

from agents import Agent, Runner, RunConfig, function_tool
from agents.models.openai_provider import OpenAIProvider

repo = MovieAPIRepository()

# Custom base URL 설정
run_config = RunConfig(
    model_provider=OpenAIProvider(base_url=os.getenv("AI_BASE_URL")),
    tracing_disabled=True,
)


@function_tool
async def get_popular_movies() -> str:
    """현재 인기 있는 영화 목록을 가져옵니다. 인기 영화, 최신 영화, 트렌드 영화를 알고 싶을 때 사용합니다."""
    use_case = GetPopularMoviesUseCase(repo)
    movies = await use_case.execute()
    lines = []
    for m in movies:
        rating = f" (평점: {m.vote_average})" if m.vote_average else ""
        lines.append(f"- [{m.id}] {m.title}{rating}")
    return "\n".join(lines)


@function_tool
async def get_movie_details(movie_id: int) -> str:
    """특정 영화의 상세 정보를 조회합니다. 영화 ID를 사용하여 제목, 줄거리, 평점 등을 확인할 수 있습니다.

    Args:
        movie_id: 조회할 영화의 ID
    """
    use_case = GetMovieDetailsUseCase(repo)
    movie = await use_case.execute(movie_id)
    return (
        f"제목: {movie.title}\n"
        f"개봉일: {movie.release_date or 'N/A'}\n"
        f"평점: {movie.vote_average or 'N/A'}\n"
        f"줄거리: {movie.overview or 'N/A'}"
    )


@function_tool
async def get_movie_credits(movie_id: int) -> str:
    """특정 영화의 출연진 및 제작진 정보를 조회합니다. 영화에 누가 출연하는지, 감독이 누구인지 알고 싶을 때 사용합니다.

    Args:
        movie_id: 조회할 영화의 ID
    """
    use_case = GetMovieCreditsUseCase(repo)
    credits = await use_case.execute(movie_id)
    lines = ["## 출연진"]
    for c in credits.cast:
        role = f" ({c.character})" if c.character else ""
        lines.append(f"- {c.name}{role}")
    lines.append("\n## 제작진")
    for c in credits.crew:
        job = f" - {c.job}" if c.job else ""
        lines.append(f"- {c.name}{job}")
    return "\n".join(lines)


movie_expert_agent = Agent(
    name="Movie Expert Agent",
    model="gpt-4o-mini",
    instructions="""당신은 영화 전문가 에이전트입니다.
사용자의 질문에 따라 적절한 도구를 선택하여 영화 정보를 제공합니다.

- 인기 영화를 물어보면 get_popular_movies를 사용하세요.
- 특정 영화의 상세 정보를 물어보면 get_movie_details를 사용하세요.
- 특정 영화의 출연진/제작진을 물어보면 get_movie_credits를 사용하세요.

항상 한국어로 친절하게 답변하세요.""",
    tools=[get_popular_movies, get_movie_details, get_movie_credits],
)

## 5. 실행

In [11]:
while True:
    query = input("질문을 입력하세요 (종료: q): ").strip()
    if not query or query.lower() == "q":
        print("종료합니다.")
        break
    result = await Runner.run(movie_expert_agent, query, run_config=run_config)
    print(f"\n{result.final_output}\n")


현재 인기 있는 영화 목록은 다음과 같습니다:

1. **Your Heart Will Be Broken** - 평점: 6.602
2. **Avatar: Fire and Ash** - 평점: 7.4
3. **Hoppers** - 평점: 7.603
4. **Crime 101** - 평점: 7.051
5. **Mike & Nick & Nick & Alice** - 평점: 6.9
6. **Shelter** - 평점: 6.798
7. **The Super Mario Galaxy Movie** - 평점: 6.962
8. **Pretty Lethal** - 평점: 6.9
9. **Humint** - 평점: 7.164
10. **GOAT** - 평점: 7.877
11. **Greenland 2: Migration** - 평점: 6.507
12. **War Machine** - 평점: 7.256
13. **The Super Mario Bros. Movie** - 평점: 7.587
14. **Project Hail Mary** - 평점: 8.189
15. **Starbright** - 평점: 7.35
16. **Scream 7** - 평점: 6.027
17. **The Passion of the Christ** - 평점: 7.538
18. **Zootopia 2** - 평점: 7.601
19. **The Shadow's Edge** - 평점: 7.2
20. **Demon Slayer: Kimetsu no Yaiba Infinity Castle** - 평점: 7.67

관심 있는 영화가 있으면 더 자세히 알려드릴 수 있습니다!


영화 ID 550에 해당하는 영화는 **"Fight Club"**입니다.

- **개봉일:** 1999년 10월 15일
- **평점:** 8.439
- **줄거리:** 불면증에 시달리는 한 남자와 미끄러운 비누 판매자가 본능적인 남성 공격성을 새로운 형태의 치료법으로 발산하게 됩니다. 이들의 개념은 모든 도시에서 underground "파이트 클럽"이